
# Tratamento da PPH 2019 para regressão de consumo energético

Este notebook produz dois dataframes compactos e reproduzíveis para estimar o **consumo médio mensal em kWh**: janeiro a junho e junho a dezembro. Junho participa dos dois períodos.

Decisões principais:

- em cada dataframe, usa somente registros com pelo menos **3 meses positivos de consumo no respectivo período**;
- evita milhares de colunas brutas, agregando horários, iluminação, equipamentos e hábitos;
- não normaliza, não aplica one-hot encoding e não treina o modelo nesta etapa;
- não usa os próprios meses de consumo como features, evitando vazamento do alvo;
- mantém município apenas para validação agrupada, não como variável preditora;
- trata ausências com zero quando representam ausência do equipamento e com mediana quando representam especificação não informada.


In [1]:

from pathlib import Path
from zipfile import ZipFile
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)


In [4]:

# O notebook procura primeiro o arquivo na mesma pasta e depois em /mnt/data.
NOME_ARQUIVO = "PPH 2019 - Banco de Dados V2.xlsx"
candidatos = [Path(NOME_ARQUIVO), Path("/mnt/data") / NOME_ARQUIVO]
ARQUIVO_XLSX = next((p for p in candidatos if p.exists()), None)
if ARQUIVO_XLSX is None:
    raise FileNotFoundError(
        f"Arquivo {NOME_ARQUIVO!r} não encontrado. Coloque-o na mesma pasta do notebook."
    )

PASTA_SAIDA = ARQUIVO_XLSX.parent
ARQUIVO_CSV_JAN_JUN = PASTA_SAIDA / "df_energia_regressao_jan_jun.csv"
ARQUIVO_CSV_JUN_DEZ = PASTA_SAIDA / "df_energia_regressao_jun_dez.csv"
ARQUIVO_DICIONARIO = PASTA_SAIDA / "dicionario_df_energia.csv"

print(f"Fonte: {ARQUIVO_XLSX}")


Fonte: PPH 2019 - Banco de Dados V2.xlsx


## 1. Seleção das colunas brutas

In [6]:

# Leitura apenas do cabeçalho com pandas. Isso permite localizar dinamicamente
# os blocos de iluminação sem carregar as 2.319 colunas completas na memória.
cabecalho = pd.read_excel(ARQUIVO_XLSX, sheet_name="Banco de Dados", nrows=0)
todas_colunas = cabecalho.columns.astype(str).tolist()

colunas_consumo = [f"P5.15.2_{mes}" for mes in range(1, 13)]

colunas_base = [
    "ENTREVISTA", "REGIAO", "UF", "MUNICIPIO", "CLASSE",
    "P4.1_1", "P4.2_1", "P4.2_2", "P4.2_3", "P4.2_4",
    "P5.1", "P5.7_1_TXT", "P5.13",
    "P7.1", "P7.1.1.3", "P7.1.1.5_1_TXT",
    "P7.2", "P7.2.1.3", "P7.2.1.5_1_TXT",
    "P7.3", "P7.3.1.3", "P7.3.1.6", "P7.3.1.8_1_TXT",
    "P8.1", "P8.1.1.2.2", "P8.1.1.3_1_TXT",
    "P8.2", "P8.2.1.3", "P8.2.1.5_1_TXT",
    "P8.3", "P8.3.1.3", "P8.3.1.5_1_TXT", "P8.3.1.6", "P8.3.1.7_1_TXT",
    "P10.8", "P10.9.2", "P10.9.3", "P10.9.4", "P10.9.5", "P10.9.D",
]

colunas_equipamentos = [
    "P9.1.2.11", "P9.1.2.13", "P9.1.2.22", "P9.1.2.23",
    "P9.1.2.36", "P9.1.2.37",
    "P9.2.2.1", "P9.2.2.2", "P9.2.2.3", "P9.2.2.4", "P9.2.2.5",
    "P9.2.2.6", "P9.2.2.8", "P9.2.2.9", "P9.2.2.10",
    "P9.2.2.11", "P9.2.2.12", "P9.2.2.14",
]

colunas_habitos = [f"P11.2.{i}" for i in range(1, 21)]
colunas_horas_ar = [f"P7.3.2.2_{h}" for h in range(24)]
colunas_meses_ar = [f"P7.3.2.3_{m}" for m in range(1, 13)]
colunas_horas_tv = [f"P8.1.1.2_{h}" for h in range(24)]
colunas_horas_lavadora = [f"P8.3.1.4_{h}" for h in range(24)]
colunas_horas_chuveiro = [f"P10.9.6_{h}" for h in range(24)]

# Quantidade de lâmpadas por tipo e quantidade de lâmpadas ligadas por hora.
regex_lampadas = re.compile(r"^P6\.1\.2\.\d+_\d+_TXT$")
regex_horas_luz = re.compile(r"^P6\.1\.3\.\d+_\d+_TXT$")
colunas_lampadas = [c for c in todas_colunas if regex_lampadas.match(c)]
colunas_horas_luz = [c for c in todas_colunas if regex_horas_luz.match(c)]

colunas_selecionadas = list(dict.fromkeys(
    colunas_base + colunas_equipamentos + colunas_habitos
    + colunas_horas_ar + colunas_meses_ar + colunas_horas_tv
    + colunas_horas_lavadora + colunas_horas_chuveiro
    + colunas_lampadas + colunas_horas_luz + colunas_consumo
))

faltantes = sorted(set(colunas_selecionadas) - set(todas_colunas))
if faltantes:
    raise KeyError(f"Colunas esperadas não encontradas: {faltantes}")

print(f"Colunas brutas selecionadas: {len(colunas_selecionadas)} de {len(todas_colunas)}")
print(f"  iluminação (quantidades): {len(colunas_lampadas)}")
print(f"  iluminação (horários): {len(colunas_horas_luz)}")


Colunas brutas selecionadas: 750 de 2319
  iluminação (quantidades): 312
  iluminação (horários): 240


## 2. Importação do XLSX para um DataFrame pandas

In [9]:

# A planilha tem mais de 60 milhões de posições de célula. O modo otimizado lê
# somente as colunas escolhidas do XML interno do XLSX e devolve um DataFrame pandas.
# Para usar estritamente pd.read_excel, altere MODO_LEITURA para "pandas".
MODO_LEITURA = "otimizado"  # "otimizado" ou "pandas"


def _numero_coluna_excel(letras: str) -> int:
    numero = 0
    for letra in letras:
        numero = numero * 26 + ord(letra) - 64
    return numero


def ler_xlsx_colunas_otimizado(arquivo: Path, colunas: list[str]) -> pd.DataFrame:
    try:
        from lxml import etree
    except ImportError as exc:
        raise ImportError(
            "O modo otimizado requer lxml. Use MODO_LEITURA='pandas' ou instale lxml."
        ) from exc

    ns = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
    regex_ref = re.compile(r"([A-Z]+)\d+")
    colunas_set = set(colunas)

    def ler_shared_strings(zip_file: ZipFile) -> list[str]:
        valores: list[str] = []
        with zip_file.open("xl/sharedStrings.xml") as stream:
            for _, elemento in etree.iterparse(stream, events=("end",), tag=ns + "si"):
                textos = elemento.xpath(
                    ".//main:t/text()",
                    namespaces={"main": ns[1:-1]},
                )
                valores.append("".join(textos))
                elemento.clear()
                while elemento.getprevious() is not None:
                    del elemento.getparent()[0]
        return valores

    def valor_celula(celula, shared_strings):
        tipo = celula.get("t")
        if tipo == "inlineStr":
            textos = celula.xpath(
                ".//main:t/text()",
                namespaces={"main": ns[1:-1]},
            )
            return "".join(textos)
        valor = celula.find(ns + "v")
        if valor is None or valor.text is None:
            return None
        texto = valor.text
        if tipo == "s":
            return shared_strings[int(texto)]
        if tipo == "b":
            return int(texto) == 1
        return texto

    registros: list[dict] = []
    with ZipFile(arquivo) as zip_file:
        shared_strings = ler_shared_strings(zip_file)
        indice_para_nome: dict[int, str] = {}

        with zip_file.open("xl/worksheets/sheet1.xml") as stream:
            contexto = etree.iterparse(stream, events=("end",), tag=ns + "row")
            for _, linha in contexto:
                numero_linha = int(linha.get("r"))

                if numero_linha == 1:
                    for celula in linha.findall(ns + "c"):
                        referencia = celula.get("r")
                        letras = regex_ref.match(referencia).group(1)
                        indice = _numero_coluna_excel(letras)
                        nome = valor_celula(celula, shared_strings)
                        if nome in colunas_set:
                            indice_para_nome[indice] = nome

                    nao_encontradas = colunas_set - set(indice_para_nome.values())
                    if nao_encontradas:
                        raise KeyError(f"Colunas não encontradas: {sorted(nao_encontradas)}")
                else:
                    registro = {nome: None for nome in colunas}
                    for celula in linha.findall(ns + "c"):
                        referencia = celula.get("r")
                        letras = regex_ref.match(referencia).group(1)
                        indice = _numero_coluna_excel(letras)
                        nome = indice_para_nome.get(indice)
                        if nome is not None:
                            registro[nome] = valor_celula(celula, shared_strings)
                    registros.append(registro)

                linha.clear()
                while linha.getprevious() is not None:
                    del linha.getparent()[0]

    return pd.DataFrame(registros, columns=colunas)


if MODO_LEITURA == "pandas":
    df_bruto = pd.read_excel(
        ARQUIVO_XLSX,
        sheet_name="Banco de Dados",
        usecols=colunas_selecionadas,
    )
else:
    df_bruto = ler_xlsx_colunas_otimizado(ARQUIVO_XLSX, colunas_selecionadas)

print(f"Formato bruto selecionado: {df_bruto.shape}")
df_bruto.head(3)


Formato bruto selecionado: (27826, 750)


,ENTREVISTA,REGIAO,UF,MUNICIPIO,CLASSE,P4.1_1,P4.2_1,P4.2_2,P4.2_3,P4.2_4,P5.1,P5.7_1_TXT,P5.13,P7.1,P7.1.1.3,P7.1.1.5_1_TXT,P7.2,P7.2.1.3,P7.2.1.5_1_TXT,P7.3,P7.3.1.3,P7.3.1.6,P7.3.1.8_1_TXT,P8.1,P8.1.1.2.2,P8.1.1.3_1_TXT,P8.2,P8.2.1.3,P8.2.1.5_1_TXT,P8.3,P8.3.1.3,P8.3.1.5_1_TXT,P8.3.1.6,P8.3.1.7_1_TXT,P10.8,P10.9.2,P10.9.3,P10.9.4,P10.9.5,P10.9.D,P9.1.2.11,P9.1.2.13,P9.1.2.22,P9.1.2.23,P9.1.2.36,P9.1.2.37,P9.2.2.1,P9.2.2.2,P9.2.2.3,P9.2.2.4,...,P6.1.3.11_10_TXT,P6.1.3.11_11_TXT,P6.1.3.11_12_TXT,P6.1.3.11_13_TXT,P6.1.3.11_14_TXT,P6.1.3.11_15_TXT,P6.1.3.11_16_TXT,P6.1.3.11_17_TXT,P6.1.3.11_18_TXT,P6.1.3.11_19_TXT,P6.1.3.11_20_TXT,P6.1.3.11_21_TXT,P6.1.3.11_22_TXT,P6.1.3.11_23_TXT,P6.1.3.12_0_TXT,P6.1.3.12_1_TXT,P6.1.3.12_2_TXT,P6.1.3.12_3_TXT,P6.1.3.12_4_TXT,P6.1.3.12_5_TXT,P6.1.3.12_6_TXT,P6.1.3.12_7_TXT,P6.1.3.12_8_TXT,P6.1.3.12_9_TXT,P6.1.3.12_10_TXT,P6.1.3.12_11_TXT,P6.1.3.12_12_TXT,P6.1.3.12_13_TXT,P6.1.3.12_14_TXT,P6.1.3.12_15_TXT,P6.1.3.12_16_TXT,P6.1.3.12_17_TXT,P6.1.3.12_18_TXT,P6.1.3.12_19_TXT,P6.1.3.12_20_TXT,P6.1.3.12_21_TXT,P6.1.3.12_22_TXT,P6.1.3.12_23_TXT,P5.15.2_1,P5.15.2_2,P5.15.2_3,P5.15.2_4,P5.15.2_5,P5.15.2_6,P5.15.2_7,P5.15.2_8,P5.15.2_9,P5.15.2_10,P5.15.2_11,P5.15.2_12
0,52641,Norte,RR,Boa Vista,4,2,0,0,2,0,1,NaN,1,1,2,10,0,NaN,NaN,2,2,20,2,1,4,3,0,NaN,NaN,1,2,2,5,3,2,6,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,52642,Norte,RR,Boa Vista,1,7,0,2,5,0,1,NaN,1,1,3,NaN,0,NaN,NaN,4,2,20,1,5,9,1,1,1,1,1,3,3,4,NaN,5,6,NaN,NaN,NaN,NaN,1,1,1,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,52643,Norte,RR,Boa Vista,5,3,0,0,2,1,1,NaN,1,1,9,8,0,NaN,NaN,2,3,22,10,1,3,8,0,NaN,NaN,1,3,1,5,3,2,6,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Limpeza, renomeação e engenharia de variáveis

In [10]:

def numerico(serie: pd.Series) -> pd.Series:
    return pd.to_numeric(serie, errors="coerce")


def contar_selecoes(df: pd.DataFrame, colunas: list[str]) -> pd.Series:
    # Nas perguntas de múltipla seleção, a célula fica preenchida quando o horário/mês foi marcado.
    return df[colunas].notna().sum(axis=1).astype("int16")


def preencher_condicional(
    serie: pd.Series,
    quantidade: pd.Series,
    valor_quando_nao_possui: float = 0.0,
) -> pd.Series:
    serie = numerico(serie)
    quantidade = numerico(quantidade).fillna(0)
    donos = (quantidade > 0) & serie.notna()
    mediana = serie.loc[donos].median()
    if pd.isna(mediana):
        mediana = 0.0
    resultado = serie.copy()
    resultado.loc[quantidade <= 0] = valor_quando_nao_possui
    resultado.loc[(quantidade > 0) & resultado.isna()] = mediana
    return resultado.astype(float)


# -----------------------------------------------------------------------------
# 1) Alvos sazonais: janeiro-junho e junho-dez (junho pertence aos dois).
# -----------------------------------------------------------------------------
consumo_mensal = df_bruto[colunas_consumo].apply(pd.to_numeric, errors="coerce")
consumo_mensal = consumo_mensal.mask(consumo_mensal <= 0)
colunas_jan_jun = [f"P5.15.2_{mes}" for mes in range(1, 7)]
colunas_jun_dez = [f"P5.15.2_{mes}" for mes in range(6, 13)]
meses_validos_jan_jun = consumo_mensal[colunas_jan_jun].notna().sum(axis=1)
meses_validos_jun_dez = consumo_mensal[colunas_jun_dez].notna().sum(axis=1)
mascara_jan_jun = meses_validos_jan_jun >= 3
mascara_jun_dez = meses_validos_jun_dez >= 3
mascara_modelo = mascara_jan_jun | mascara_jun_dez

base = df_bruto.loc[mascara_modelo].copy()
consumo_mensal = consumo_mensal.loc[mascara_modelo]
meses_validos_jan_jun = meses_validos_jan_jun.loc[mascara_modelo]
meses_validos_jun_dez = meses_validos_jun_dez.loc[mascara_modelo]
mascara_jan_jun = mascara_jan_jun.loc[mascara_modelo]
mascara_jun_dez = mascara_jun_dez.loc[mascara_modelo]

df_modelo = pd.DataFrame(index=base.index)

# -----------------------------------------------------------------------------
# 2) Identificação e contexto. Município é mantido somente para validação agrupada.
# -----------------------------------------------------------------------------
df_modelo["id_entrevista"] = numerico(base["ENTREVISTA"]).astype("int64")
df_modelo["regiao"] = base["REGIAO"].fillna("Nao informado").astype(str).str.strip()
df_modelo["uf"] = base["UF"].fillna("NA").astype(str).str.strip()
df_modelo["grupo_validacao_municipio"] = (
    df_modelo["uf"] + "__" + base["MUNICIPIO"].fillna("Nao informado").astype(str).str.strip()
)

df_modelo["classe_social"] = (
    numerico(base["CLASSE"])
    .map({1: "A", 2: "B1", 3: "B2", 4: "C1", 5: "C2", 6: "D-E"})
    .fillna("Nao informado")
)
df_modelo["tipo_imovel"] = (
    numerico(base["P5.1"])
    .map({1: "Casa", 2: "Apartamento", 3: "Quarto ou comodo"})
    .fillna("Nao informado")
)

atividade = numerico(base["P5.13"]).map({1: 0, 2: 1})
df_modelo["atividade_comercial_ausente"] = atividade.isna().astype("int8")
df_modelo["tem_atividade_comercial"] = atividade.fillna(0).astype("int8")

# -----------------------------------------------------------------------------
# 3) Moradores e imóvel.
# -----------------------------------------------------------------------------
df_modelo["moradores_habituais"] = numerico(base["P4.1_1"]).fillna(0).clip(lower=0)
df_modelo["criancas_ate_10"] = numerico(base["P4.2_1"]).fillna(0).clip(lower=0)
df_modelo["adolescentes_11_18"] = numerico(base["P4.2_2"]).fillna(0).clip(lower=0)
df_modelo["adultos_19_65"] = numerico(base["P4.2_3"]).fillna(0).clip(lower=0)
df_modelo["idosos_acima_65"] = numerico(base["P4.2_4"]).fillna(0).clip(lower=0)

area = numerico(base["P5.7_1_TXT"])
area = area.where(area.between(5, 2000))
df_modelo["area_m2_ausente"] = area.isna().astype("int8")

# Imputação hierárquica: mediana por região e tipo de imóvel; depois mediana global.
chaves_area = pd.DataFrame({
    "regiao": df_modelo["regiao"],
    "tipo_imovel": df_modelo["tipo_imovel"],
    "area": area,
})
mediana_grupo = chaves_area.groupby(["regiao", "tipo_imovel"])["area"].transform("median")
area = area.fillna(mediana_grupo).fillna(area.median())
df_modelo["area_m2"] = area.astype(float)

# -----------------------------------------------------------------------------
# 4) Quantidades de equipamentos.
# -----------------------------------------------------------------------------
mapa_contagens = {
    "P7.1": "qtd_refrigeradores",
    "P7.2": "qtd_freezers",
    "P7.3": "qtd_ar_condicionado",
    "P8.1": "qtd_televisores",
    "P8.2": "qtd_microondas",
    "P8.3": "qtd_maquinas_lavar",
    "P10.8": "qtd_chuveiros",
    "P9.1.2.11": "qtd_fogoes_eletricos",
    "P9.1.2.13": "qtd_fritadeiras_sem_oleo",
    "P9.1.2.22": "qtd_modems",
    "P9.1.2.23": "qtd_roteadores",
    "P9.1.2.36": "qtd_filtros_piscina",
    "P9.1.2.37": "qtd_bombas_agua",
    "P9.2.2.1": "qtd_fornos_eletricos",
    "P9.2.2.2": "qtd_lava_loucas",
    "P9.2.2.3": "qtd_ferros_secos",
    "P9.2.2.4": "qtd_ferros_vapor",
    "P9.2.2.5": "qtd_ferros_vapor_sem_vapor",
    "P9.2.2.6": "qtd_secadoras_aquecimento",
    "P9.2.2.8": "qtd_aquecedores_ambiente",
    "P9.2.2.9": "qtd_ventiladores_teto",
    "P9.2.2.10": "qtd_ventiladores_portateis",
    "P9.2.2.11": "qtd_videogames",
    "P9.2.2.12": "qtd_notebooks",
    "P9.2.2.14": "qtd_computadores",
}

for coluna_origem, coluna_destino in mapa_contagens.items():
    df_modelo[coluna_destino] = numerico(base[coluna_origem]).fillna(0).clip(lower=0)

colunas_contagem = list(mapa_contagens.values())
df_modelo["total_equipamentos_relevantes"] = df_modelo[colunas_contagem].sum(axis=1)

df_modelo["total_equipamentos_alta_potencia"] = df_modelo[[
    "qtd_ar_condicionado", "qtd_maquinas_lavar", "qtd_fogoes_eletricos",
    "qtd_fritadeiras_sem_oleo", "qtd_filtros_piscina", "qtd_bombas_agua",
    "qtd_fornos_eletricos", "qtd_lava_loucas", "qtd_secadoras_aquecimento",
    "qtd_aquecedores_ambiente",
]].sum(axis=1)

# -----------------------------------------------------------------------------
# 5) Especificações e uso dos principais equipamentos.
# -----------------------------------------------------------------------------
qtd_ref = df_modelo["qtd_refrigeradores"]
qtd_freezer = df_modelo["qtd_freezers"]
qtd_ar = df_modelo["qtd_ar_condicionado"]
qtd_tv = df_modelo["qtd_televisores"]
qtd_micro = df_modelo["qtd_microondas"]
qtd_lavadora = df_modelo["qtd_maquinas_lavar"]

cap_ref = numerico(base["P7.1.1.3"]).map({1: 150, 2: 250, 3: 350, 4: 450, 5: 550})
cap_freezer = numerico(base["P7.2.1.3"]).map({1: 120, 2: 190, 3: 270, 4: 350})
btu_ar = numerico(base["P7.3.1.3"]).map({1: 7000, 2: 9000, 3: 11000, 4: 15000, 5: 21000, 6: 28000})
tamanho_tv = numerico(base["P8.1.1.2.2"]).map({1: 18, 2: 25, 3: 35, 4: 45, 5: 55})
cap_lavadora = numerico(base["P8.3.1.3"]).map({1: 6, 2: 8, 3: 11})

# Capacidade total aproxima a característica do aparelho informado multiplicada pela quantidade.
df_modelo["capacidade_refrigeracao_litros_proxy"] = (
    preencher_condicional(cap_ref, qtd_ref) * qtd_ref
    + preencher_condicional(cap_freezer, qtd_freezer) * qtd_freezer
)
df_modelo["capacidade_total_ar_btu_proxy"] = preencher_condicional(btu_ar, qtd_ar) * qtd_ar
df_modelo["tamanho_tv_polegadas_proxy"] = preencher_condicional(tamanho_tv, qtd_tv)
df_modelo["capacidade_lavadora_kg_proxy"] = preencher_condicional(cap_lavadora, qtd_lavadora)

df_modelo["idade_refrigerador_anos"] = preencher_condicional(base["P7.1.1.5_1_TXT"], qtd_ref)
df_modelo["idade_freezer_anos"] = preencher_condicional(base["P7.2.1.5_1_TXT"], qtd_freezer)
df_modelo["idade_ar_condicionado_anos"] = preencher_condicional(base["P7.3.1.8_1_TXT"], qtd_ar)
df_modelo["idade_tv_anos"] = preencher_condicional(base["P8.1.1.3_1_TXT"], qtd_tv)
df_modelo["idade_microondas_anos"] = preencher_condicional(base["P8.2.1.5_1_TXT"], qtd_micro)
df_modelo["idade_maquina_lavar_anos"] = preencher_condicional(base["P8.3.1.7_1_TXT"], qtd_lavadora)

temperatura_ar = numerico(base["P7.3.1.6"]).where(lambda s: s.between(14, 30))
df_modelo["temperatura_ar_c"] = preencher_condicional(temperatura_ar, qtd_ar)

df_modelo["meses_uso_ar_ano"] = contar_selecoes(base, colunas_meses_ar)
df_modelo["horas_uso_ar_dia"] = contar_selecoes(base, colunas_horas_ar)
df_modelo["horas_uso_ar_pico"] = base[[f"P7.3.2.2_{h}" for h in range(18, 22)]].notna().sum(axis=1)

df_modelo["horas_uso_tv_dia"] = contar_selecoes(base, colunas_horas_tv)
df_modelo["horas_uso_tv_pico"] = base[[f"P8.1.1.2_{h}" for h in range(18, 22)]].notna().sum(axis=1)

df_modelo["horas_uso_lavadora"] = contar_selecoes(base, colunas_horas_lavadora)
df_modelo["horas_uso_lavadora_pico"] = base[[f"P8.3.1.4_{h}" for h in range(18, 22)]].notna().sum(axis=1)

lavagens_por_uso = numerico(base["P8.3.1.5_1_TXT"])
frequencia_lavadora_semana = numerico(base["P8.3.1.6"]).map({
    1: 6.5, 2: 4.5, 3: 2.5, 4: 1.0, 5: 0.5, 6: 0.25, 7: 0.0,
})
lavagens_por_uso = preencher_condicional(lavagens_por_uso, qtd_lavadora)
frequencia_lavadora_semana = preencher_condicional(frequencia_lavadora_semana, qtd_lavadora)
df_modelo["lavagens_estimadas_semana"] = lavagens_por_uso * frequencia_lavadora_semana

# Micro-ondas: ponto médio das faixas de minutos no dia de utilização.
tempo_micro = numerico(base["P8.2.1.3"]).replace(98, np.nan).map({1: 5, 2: 20, 3: 45, 4: 75, 5: 100})
df_modelo["minutos_microondas_dia_uso"] = preencher_condicional(tempo_micro, qtd_micro)

# Chuveiro.
fonte_chuveiro = numerico(base["P10.9.2"])
df_modelo["chuveiro_eletrico"] = (fonte_chuveiro == 1).astype("int8")
potencia_chuveiro = numerico(base["P10.9.4"]).map({1: 4000, 2: 5250, 3: 6750, 4: 8000})
duracao_banho = numerico(base["P10.9.D"]).map({1: 5, 2: 8, 3: 15, 4: 25})
banhos_dia = numerico(base["P10.9.5"])

# Sem aquecimento elétrico, potência elétrica e proxy de consumo do banho são zero.
potencia_chuveiro = potencia_chuveiro.where(df_modelo["chuveiro_eletrico"].eq(1), 0)
duracao_banho = duracao_banho.where(df_modelo["chuveiro_eletrico"].eq(1), 0)
banhos_dia = banhos_dia.where(df_modelo["chuveiro_eletrico"].eq(1), 0)

for serie, nome in [
    (potencia_chuveiro, "potencia_chuveiro_w"),
    (duracao_banho, "duracao_media_banho_min"),
    (banhos_dia, "banhos_por_dia"),
]:
    mediana = serie[serie > 0].median()
    df_modelo[nome] = serie.fillna(0 if pd.isna(mediana) else mediana)

df_modelo["consumo_chuveiro_kwh_dia_proxy"] = (
    df_modelo["potencia_chuveiro_w"] / 1000
    * df_modelo["banhos_por_dia"]
    * df_modelo["duracao_media_banho_min"] / 60
)
df_modelo["horas_uso_chuveiro"] = contar_selecoes(base, colunas_horas_chuveiro)
df_modelo["horas_uso_chuveiro_pico"] = base[[f"P10.9.6_{h}" for h in range(18, 22)]].notna().sum(axis=1)

# -----------------------------------------------------------------------------
# 6) Iluminação agregada.
# -----------------------------------------------------------------------------
quantidades_lampadas = base[colunas_lampadas].apply(pd.to_numeric, errors="coerce").fillna(0)

# Tipo da lâmpada é o número depois do último '_' e antes de '_TXT'.
def tipo_lampada(coluna: str) -> int:
    return int(re.search(r"_(\d+)_TXT$", coluna).group(1))

# Valores aproximados apenas como proxy de potência instalada.
potencia_tipo_w = {
    1: 60, 2: 40, 3: 60, 4: 100, 5: 150,
    6: 18, 7: 12, 8: 15, 9: 21, 10: 30,
    11: 80, 12: 40, 13: 20,
    14: 10, 15: 8, 16: 10, 17: 14, 18: 18,
    19: 36, 20: 18, 21: 9,
    22: 50, 23: 7, 24: 35, 25: 5, 26: 40,
}
tipos_led = {14, 15, 16, 17, 18, 19, 20, 21, 23, 25}
colunas_led = [c for c in colunas_lampadas if tipo_lampada(c) in tipos_led]

df_modelo["total_lampadas"] = quantidades_lampadas.sum(axis=1)
df_modelo["total_lampadas_led"] = quantidades_lampadas[colunas_led].sum(axis=1)
df_modelo["percentual_lampadas_led"] = np.where(
    df_modelo["total_lampadas"] > 0,
    df_modelo["total_lampadas_led"] / df_modelo["total_lampadas"],
    0,
)

df_modelo["potencia_iluminacao_instalada_w_proxy"] = sum(
    quantidades_lampadas[coluna] * potencia_tipo_w[tipo_lampada(coluna)]
    for coluna in colunas_lampadas
)

horas_luz = base[colunas_horas_luz].apply(pd.to_numeric, errors="coerce").fillna(0)
df_modelo["lampada_horas_dia_proxy"] = horas_luz.sum(axis=1)
colunas_luz_pico = [c for c in colunas_horas_luz if int(re.search(r"_(\d+)_TXT$", c).group(1)) in range(18, 22)]
df_modelo["lampada_horas_pico_proxy"] = horas_luz[colunas_luz_pico].sum(axis=1)

# -----------------------------------------------------------------------------
# 7) Hábitos de eficiência.
# -----------------------------------------------------------------------------
mapa_habito = {1: 1.0, 2: 2 / 3, 3: 1 / 3, 9: 0.0, 10: np.nan}
habitos = pd.DataFrame(index=base.index)
for i in range(1, 21):
    habitos[i] = numerico(base[f"P11.2.{i}"]).map(mapa_habito)

subgrupos_habitos = {
    "indice_habitos_geral": list(range(1, 21)),
    "indice_habitos_standby": [2, 3],
    "indice_habitos_climatizacao": [4, 5],
    "indice_habitos_banho": [6, 7],
    "indice_habitos_refrigeracao": [8, 9, 10, 11, 12],
    "indice_habitos_lavanderia": [13, 14, 15, 16],
    "indice_habitos_iluminacao": [17, 18, 19],
}

for nome, itens in subgrupos_habitos.items():
    indice = habitos[itens].mean(axis=1)
    df_modelo[nome] = indice.fillna(indice.median()).fillna(0.5)

df_modelo["proporcao_habitos_nao_aplicaveis"] = habitos.isna().mean(axis=1)

# -----------------------------------------------------------------------------
# 8) Variáveis agregadas de horário de pico.
# -----------------------------------------------------------------------------
df_modelo["horas_alto_consumo_pico_proxy"] = (
    df_modelo["horas_uso_ar_pico"]
    + df_modelo["horas_uso_lavadora_pico"]
    + df_modelo["horas_uso_chuveiro_pico"]
)
df_modelo["uso_horario_pico"] = (
    (df_modelo["horas_alto_consumo_pico_proxy"] > 0)
    | (df_modelo["lampada_horas_pico_proxy"] > 0)
).astype("int8")

# -----------------------------------------------------------------------------
# 9) Controle de qualidade e alvo final.
# -----------------------------------------------------------------------------
# As colunas de controle e alvo são adicionadas separadamente mais abaixo.

# Ordenação por grupos: controle/contexto, imóvel, equipamentos, uso, hábitos e alvo.
colunas_inicio = [
    "id_entrevista", "grupo_validacao_municipio", "regiao", "uf",
    "classe_social", "tipo_imovel", "tem_atividade_comercial",
    "atividade_comercial_ausente", "moradores_habituais", "criancas_ate_10",
    "adolescentes_11_18", "adultos_19_65", "idosos_acima_65",
    "area_m2", "area_m2_ausente",
]
colunas_fim = [
    "indice_habitos_geral", "indice_habitos_standby",
    "indice_habitos_climatizacao", "indice_habitos_banho",
    "indice_habitos_refrigeracao", "indice_habitos_lavanderia",
    "indice_habitos_iluminacao", "proporcao_habitos_nao_aplicaveis",
]
colunas_meio = [c for c in df_modelo.columns if c not in colunas_inicio + colunas_fim]
df_modelo = df_modelo[colunas_inicio + colunas_meio + colunas_fim]

# Conversão de contagens inteiras para tipos compactos.
for coluna in df_modelo.columns:
    if coluna.startswith("qtd_") or coluna in {
        "total_equipamentos_relevantes", "total_equipamentos_alta_potencia",
        "total_lampadas", "total_lampadas_led", "meses_uso_ar_ano",
        "horas_uso_ar_dia", "horas_uso_ar_pico", "horas_uso_tv_dia",
        "horas_uso_tv_pico", "horas_uso_lavadora", "horas_uso_lavadora_pico",
        "horas_uso_chuveiro", "horas_uso_chuveiro_pico",
        "horas_alto_consumo_pico_proxy",
    }:
        df_modelo[coluna] = pd.to_numeric(df_modelo[coluna], errors="coerce").fillna(0)

# Dois dataframes finais, cada um com seu próprio filtro, controle e alvo.
df_jan_jun = df_modelo.loc[mascara_jan_jun].copy()
df_jan_jun["controle_meses_consumo_validos"] = meses_validos_jan_jun.loc[mascara_jan_jun].astype("int8")
df_jan_jun["consumo_medio_kwh"] = consumo_mensal.loc[mascara_jan_jun, colunas_jan_jun].mean(axis=1)
df_jan_jun = df_jan_jun.reset_index(drop=True)

df_jun_dez = df_modelo.loc[mascara_jun_dez].copy()
df_jun_dez["controle_meses_consumo_validos"] = meses_validos_jun_dez.loc[mascara_jun_dez].astype("int8")
df_jun_dez["consumo_medio_kwh"] = consumo_mensal.loc[mascara_jun_dez, colunas_jun_dez].mean(axis=1)
df_jun_dez = df_jun_dez.reset_index(drop=True)

# Validações finais.
for nome, dataframe in {"jan_jun": df_jan_jun, "jun_dez": df_jun_dez}.items():
    assert not dataframe["id_entrevista"].duplicated().any(), f"Há IDs duplicados em {nome}."
    assert dataframe["consumo_medio_kwh"].gt(0).all(), f"Alvo inválido em {nome}."
    assert dataframe.isna().sum().sum() == 0, f"Ainda existem valores nulos em {nome}."
    print(f"{nome}: {len(dataframe):,} linhas, {dataframe.shape[1]} colunas")

df_jan_jun.head()


jan_jun: 2,402 linhas, 87 colunas
jun_dez: 2,548 linhas, 87 colunas


,id_entrevista,grupo_validacao_municipio,regiao,uf,classe_social,tipo_imovel,tem_atividade_comercial,atividade_comercial_ausente,moradores_habituais,criancas_ate_10,adolescentes_11_18,adultos_19_65,idosos_acima_65,area_m2,area_m2_ausente,qtd_refrigeradores,qtd_freezers,qtd_ar_condicionado,qtd_televisores,qtd_microondas,qtd_maquinas_lavar,qtd_chuveiros,qtd_fogoes_eletricos,qtd_fritadeiras_sem_oleo,qtd_modems,qtd_roteadores,qtd_filtros_piscina,qtd_bombas_agua,qtd_fornos_eletricos,qtd_lava_loucas,qtd_ferros_secos,qtd_ferros_vapor,qtd_ferros_vapor_sem_vapor,qtd_secadoras_aquecimento,qtd_aquecedores_ambiente,qtd_ventiladores_teto,qtd_ventiladores_portateis,qtd_videogames,qtd_notebooks,qtd_computadores,total_equipamentos_relevantes,total_equipamentos_alta_potencia,capacidade_refrigeracao_litros_proxy,capacidade_total_ar_btu_proxy,tamanho_tv_polegadas_proxy,capacidade_lavadora_kg_proxy,idade_refrigerador_anos,idade_freezer_anos,idade_ar_condicionado_anos,idade_tv_anos,idade_microondas_anos,idade_maquina_lavar_anos,temperatura_ar_c,meses_uso_ar_ano,horas_uso_ar_dia,horas_uso_ar_pico,horas_uso_tv_dia,horas_uso_tv_pico,horas_uso_lavadora,horas_uso_lavadora_pico,lavagens_estimadas_semana,minutos_microondas_dia_uso,chuveiro_eletrico,potencia_chuveiro_w,duracao_media_banho_min,banhos_por_dia,consumo_chuveiro_kwh_dia_proxy,horas_uso_chuveiro,horas_uso_chuveiro_pico,total_lampadas,total_lampadas_led,percentual_lampadas_led,potencia_iluminacao_instalada_w_proxy,lampada_horas_dia_proxy,lampada_horas_pico_proxy,horas_alto_consumo_pico_proxy,uso_horario_pico,indice_habitos_geral,indice_habitos_standby,indice_habitos_climatizacao,indice_habitos_banho,indice_habitos_refrigeracao,indice_habitos_lavanderia,indice_habitos_iluminacao,proporcao_habitos_nao_aplicaveis,controle_meses_consumo_validos,consumo_medio_kwh
0,52666,RR__Boa Vista,Norte,RR,C1,Casa,0,0,4,1,1,2,0,12.0,0,1,0,1,2,0,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,8,2,550.0,15000.0,45.0,11.0,8.0,0.0,2.0,4.0,0.0,6.0,23.0,12,4,2,8,4,1,0,5.0,0.0,0,0.0,0.0,0.0,0.0,0,0,12.0,10.0,0.833333,136.0,9.0,9.0,2,1,0.907407,0.666667,1.000000,0.833333,1.000000,0.750000,1.000000,0.1,6,307.500000
1,52669,RR__Boa Vista,Norte,RR,D-E,Casa,0,0,4,0,2,2,0,49.0,1,1,0,2,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,6,2,250.0,30000.0,35.0,0.0,5.0,0.0,5.0,4.0,0.0,0.0,22.0,12,0,0,1,0,0,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0,9.0,3.0,0.333333,136.0,12.0,9.0,0,1,0.685185,0.666667,0.666667,0.833333,0.600000,0.500000,1.000000,0.1,6,58.500000
2,52687,RR__Boa Vista,Norte,RR,D-E,Casa,0,0,3,0,1,2,0,14.0,0,1,0,1,2,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,7,2,150.0,11000.0,45.0,0.0,4.0,0.0,3.0,3.0,0.0,0.0,16.0,11,0,0,4,3,0,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0,6.0,1.0,0.166667,310.0,17.0,13.0,0,1,0.407407,0.666667,1.000000,0.833333,0.133333,0.083333,0.777778,0.1,5,197.400000
3,52697,RR__Boa Vista,Norte,RR,B1,Casa,0,0,4,0,0,4,0,300.0,0,1,0,3,2,1,1,2,2,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,2,1,18,6,550.0,27000.0,35.0,11.0,4.0,0.0,3.0,5.0,2.0,6.0,22.0,0,0,0,0,0,0,0,2.5,5.0,0,0.0,0.0,0.0,0.0,0,0,18.0,14.0,0.777778,208.0,45.0,28.0,0,1,0.851852,0.833333,1.000000,0.833333,0.600000,1.000000,1.000000,0.1,6,997.333333
4,52700,RR__Boa Vista,Norte,RR,B1,Casa,0,0,3,1,0,2,0,49.0,1,1,0,2,1,1,1,3,1,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,2,0,15,4,450.0,30000.0,45.0,8.0,2.0,0.0,10.0,2.0,8.0,4.0,26.0,12,0,0,0,0,0,0,2.5,5.0,0,0.0,0.0,0.0,0.0,0,0,12.0,7.0,0.583333,160.0,24.0,24.0,0,1,0.816667,0.666667,0.666667,1.000000,0.600000,0.916667,1.000000,0.0,6,703.666667


## 4. Exportação do dataframe e do dicionário

In [12]:

# Dicionário das colunas e papel recomendado no treinamento.
def grupo_coluna(coluna: str) -> str:
    if coluna in {"id_entrevista", "grupo_validacao_municipio", "controle_meses_consumo_validos"}:
        return "controle"
    if coluna == "consumo_medio_kwh":
        return "alvo"
    if coluna in {
        "regiao", "uf", "classe_social", "tipo_imovel", "tem_atividade_comercial",
        "atividade_comercial_ausente", "moradores_habituais", "criancas_ate_10",
        "adolescentes_11_18", "adultos_19_65", "idosos_acima_65", "area_m2",
        "area_m2_ausente",
    }:
        return "domicilio_contexto"
    if "habitos" in coluna:
        return "habitos"
    if "hora" in coluna or "banho" in coluna or "lavagens" in coluna or "meses_uso" in coluna:
        return "uso_horarios"
    if "lampada" in coluna or "iluminacao" in coluna:
        return "iluminacao"
    return "equipamentos"


def uso_modelo(coluna: str) -> str:
    if coluna == "consumo_medio_kwh":
        return "y_alvo"
    if coluna in {"id_entrevista", "grupo_validacao_municipio", "controle_meses_consumo_validos"}:
        return "controle_excluir_de_X"
    return "feature_X"


descricoes_especiais = {
    "id_entrevista": "Identificador único do registro.",
    "grupo_validacao_municipio": "UF e município para GroupKFold; não usar como feature.",
    "area_m2": "Área imputada pela mediana de região e tipo de imóvel.",
    "area_m2_ausente": "Indica que a área original estava ausente ou inválida.",
    "total_equipamentos_relevantes": "Soma das quantidades de equipamentos selecionados.",
    "total_equipamentos_alta_potencia": "Soma de equipamentos com maior potencial de consumo.",
    "percentual_lampadas_led": "Proporção de lâmpadas LED sobre o total informado.",
    "lampada_horas_dia_proxy": "Soma do número de lâmpadas ligadas em cada hora do dia.",
    "lampada_horas_pico_proxy": "Lâmpada-horas entre 18h e 21h.",
    "consumo_chuveiro_kwh_dia_proxy": "Estimativa diária baseada em potência, banhos e duração.",
    "uso_horario_pico": "Indica uso de cargas relevantes ou iluminação entre 18h e 21h.",
    "controle_meses_consumo_validos": "Quantidade de meses positivos usados para formar o alvo do período; excluir de X.",
    "consumo_medio_kwh": "Alvo de regressão: média dos meses positivos do período, com mínimo de 3 meses.",
}

registros_dicionario = []
for coluna in df_jan_jun.columns:
    registros_dicionario.append({
        "coluna": coluna,
        "grupo": grupo_coluna(coluna),
        "uso_no_modelo": uso_modelo(coluna),
        "tipo_dado": str(df_jan_jun[coluna].dtype),
        "descricao": descricoes_especiais.get(
            coluna,
            coluna.replace("_", " ").capitalize(),
        ),
    })

df_dicionario = pd.DataFrame(registros_dicionario)

df_jan_jun.to_csv(ARQUIVO_CSV_JAN_JUN, index=False, encoding="utf-8")
df_jun_dez.to_csv(ARQUIVO_CSV_JUN_DEZ, index=False, encoding="utf-8")
df_dicionario.to_csv(ARQUIVO_DICIONARIO, index=False, encoding="utf-8")

print(f"Dataframe janeiro-junho exportado: {ARQUIVO_CSV_JAN_JUN}")
print(f"Dataframe junho-dezembro exportado: {ARQUIVO_CSV_JUN_DEZ}")
print(f"Dicionário exportado: {ARQUIVO_DICIONARIO}")
print(f"Tamanhos: jan-jun {df_jan_jun.shape}; jun-dez {df_jun_dez.shape}")


Dataframe janeiro-junho exportado: df_energia_regressao_jan_jun.csv
Dataframe junho-dezembro exportado: df_energia_regressao_jun_dez.csv
Dicionário exportado: dicionario_df_energia.csv
Tamanhos: jan-jun (2402, 87); jun-dez (2548, 87)


## 5. Validação do resultado

In [14]:
df_jan_jun.head()

,id_entrevista,grupo_validacao_municipio,regiao,uf,classe_social,tipo_imovel,tem_atividade_comercial,atividade_comercial_ausente,moradores_habituais,criancas_ate_10,adolescentes_11_18,adultos_19_65,idosos_acima_65,area_m2,area_m2_ausente,qtd_refrigeradores,qtd_freezers,qtd_ar_condicionado,qtd_televisores,qtd_microondas,qtd_maquinas_lavar,qtd_chuveiros,qtd_fogoes_eletricos,qtd_fritadeiras_sem_oleo,qtd_modems,qtd_roteadores,qtd_filtros_piscina,qtd_bombas_agua,qtd_fornos_eletricos,qtd_lava_loucas,qtd_ferros_secos,qtd_ferros_vapor,qtd_ferros_vapor_sem_vapor,qtd_secadoras_aquecimento,qtd_aquecedores_ambiente,qtd_ventiladores_teto,qtd_ventiladores_portateis,qtd_videogames,qtd_notebooks,qtd_computadores,total_equipamentos_relevantes,total_equipamentos_alta_potencia,capacidade_refrigeracao_litros_proxy,capacidade_total_ar_btu_proxy,tamanho_tv_polegadas_proxy,capacidade_lavadora_kg_proxy,idade_refrigerador_anos,idade_freezer_anos,idade_ar_condicionado_anos,idade_tv_anos,idade_microondas_anos,idade_maquina_lavar_anos,temperatura_ar_c,meses_uso_ar_ano,horas_uso_ar_dia,horas_uso_ar_pico,horas_uso_tv_dia,horas_uso_tv_pico,horas_uso_lavadora,horas_uso_lavadora_pico,lavagens_estimadas_semana,minutos_microondas_dia_uso,chuveiro_eletrico,potencia_chuveiro_w,duracao_media_banho_min,banhos_por_dia,consumo_chuveiro_kwh_dia_proxy,horas_uso_chuveiro,horas_uso_chuveiro_pico,total_lampadas,total_lampadas_led,percentual_lampadas_led,potencia_iluminacao_instalada_w_proxy,lampada_horas_dia_proxy,lampada_horas_pico_proxy,horas_alto_consumo_pico_proxy,uso_horario_pico,indice_habitos_geral,indice_habitos_standby,indice_habitos_climatizacao,indice_habitos_banho,indice_habitos_refrigeracao,indice_habitos_lavanderia,indice_habitos_iluminacao,proporcao_habitos_nao_aplicaveis,controle_meses_consumo_validos,consumo_medio_kwh
0,52666,RR__Boa Vista,Norte,RR,C1,Casa,0,0,4,1,1,2,0,12.0,0,1,0,1,2,0,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,8,2,550.0,15000.0,45.0,11.0,8.0,0.0,2.0,4.0,0.0,6.0,23.0,12,4,2,8,4,1,0,5.0,0.0,0,0.0,0.0,0.0,0.0,0,0,12.0,10.0,0.833333,136.0,9.0,9.0,2,1,0.907407,0.666667,1.000000,0.833333,1.000000,0.750000,1.000000,0.1,6,307.500000
1,52669,RR__Boa Vista,Norte,RR,D-E,Casa,0,0,4,0,2,2,0,49.0,1,1,0,2,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,6,2,250.0,30000.0,35.0,0.0,5.0,0.0,5.0,4.0,0.0,0.0,22.0,12,0,0,1,0,0,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0,9.0,3.0,0.333333,136.0,12.0,9.0,0,1,0.685185,0.666667,0.666667,0.833333,0.600000,0.500000,1.000000,0.1,6,58.500000
2,52687,RR__Boa Vista,Norte,RR,D-E,Casa,0,0,3,0,1,2,0,14.0,0,1,0,1,2,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,7,2,150.0,11000.0,45.0,0.0,4.0,0.0,3.0,3.0,0.0,0.0,16.0,11,0,0,4,3,0,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0,6.0,1.0,0.166667,310.0,17.0,13.0,0,1,0.407407,0.666667,1.000000,0.833333,0.133333,0.083333,0.777778,0.1,5,197.400000
3,52697,RR__Boa Vista,Norte,RR,B1,Casa,0,0,4,0,0,4,0,300.0,0,1,0,3,2,1,1,2,2,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,2,1,18,6,550.0,27000.0,35.0,11.0,4.0,0.0,3.0,5.0,2.0,6.0,22.0,0,0,0,0,0,0,0,2.5,5.0,0,0.0,0.0,0.0,0.0,0,0,18.0,14.0,0.777778,208.0,45.0,28.0,0,1,0.851852,0.833333,1.000000,0.833333,0.600000,1.000000,1.000000,0.1,6,997.333333
4,52700,RR__Boa Vista,Norte,RR,B1,Casa,0,0,3,1,0,2,0,49.0,1,1,0,2,1,1,1,3,1,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,2,0,15,4,450.0,30000.0,45.0,8.0,2.0,0.0,10.0,2.0,8.0,4.0,26.0,12,0,0,0,0,0,0,2.5,5.0,0,0.0,0.0,0.0,0.0,0,0,12.0,7.0,0.583333,160.0,24.0,24.0,0,1,0.816667,0.666667,0.666667,1.000000,0.600000,0.916667,1.000000,0.0,6,703.666667


In [15]:
df_jun_dez.head()

,id_entrevista,grupo_validacao_municipio,regiao,uf,classe_social,tipo_imovel,tem_atividade_comercial,atividade_comercial_ausente,moradores_habituais,criancas_ate_10,adolescentes_11_18,adultos_19_65,idosos_acima_65,area_m2,area_m2_ausente,qtd_refrigeradores,qtd_freezers,qtd_ar_condicionado,qtd_televisores,qtd_microondas,qtd_maquinas_lavar,qtd_chuveiros,qtd_fogoes_eletricos,qtd_fritadeiras_sem_oleo,qtd_modems,qtd_roteadores,qtd_filtros_piscina,qtd_bombas_agua,qtd_fornos_eletricos,qtd_lava_loucas,qtd_ferros_secos,qtd_ferros_vapor,qtd_ferros_vapor_sem_vapor,qtd_secadoras_aquecimento,qtd_aquecedores_ambiente,qtd_ventiladores_teto,qtd_ventiladores_portateis,qtd_videogames,qtd_notebooks,qtd_computadores,total_equipamentos_relevantes,total_equipamentos_alta_potencia,capacidade_refrigeracao_litros_proxy,capacidade_total_ar_btu_proxy,tamanho_tv_polegadas_proxy,capacidade_lavadora_kg_proxy,idade_refrigerador_anos,idade_freezer_anos,idade_ar_condicionado_anos,idade_tv_anos,idade_microondas_anos,idade_maquina_lavar_anos,temperatura_ar_c,meses_uso_ar_ano,horas_uso_ar_dia,horas_uso_ar_pico,horas_uso_tv_dia,horas_uso_tv_pico,horas_uso_lavadora,horas_uso_lavadora_pico,lavagens_estimadas_semana,minutos_microondas_dia_uso,chuveiro_eletrico,potencia_chuveiro_w,duracao_media_banho_min,banhos_por_dia,consumo_chuveiro_kwh_dia_proxy,horas_uso_chuveiro,horas_uso_chuveiro_pico,total_lampadas,total_lampadas_led,percentual_lampadas_led,potencia_iluminacao_instalada_w_proxy,lampada_horas_dia_proxy,lampada_horas_pico_proxy,horas_alto_consumo_pico_proxy,uso_horario_pico,indice_habitos_geral,indice_habitos_standby,indice_habitos_climatizacao,indice_habitos_banho,indice_habitos_refrigeracao,indice_habitos_lavanderia,indice_habitos_iluminacao,proporcao_habitos_nao_aplicaveis,controle_meses_consumo_validos,consumo_medio_kwh
0,52647,RR__Boa Vista,Norte,RR,C1,Casa,0,0,7,3,0,4,0,49.0,1,1,0,1,2,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,3,0,0,3,13,1,350.0,9000.0,35.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,22.0,2,9,3,0,0,0,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0,2.0,1.0,0.500000,26.0,4.0,4.0,3,1,0.981481,1.000000,1.000000,0.833333,1.000000,1.000000,1.000000,0.1,5,486.0
1,52660,RR__Boa Vista,Norte,RR,C2,Casa,0,0,3,1,0,2,0,10.0,0,2,0,0,1,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,7,0,900.0,0.0,35.0,0.0,1.0,0.0,0.0,3.0,0.0,0.0,0.0,0,0,0,1,0,0,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0,12.0,2.0,0.166667,371.0,18.0,14.0,0,1,0.708333,1.000000,1.000000,0.833333,0.733333,0.250000,0.888889,0.2,5,131.0
2,52666,RR__Boa Vista,Norte,RR,C1,Casa,0,0,4,1,1,2,0,12.0,0,1,0,1,2,0,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,8,2,550.0,15000.0,45.0,11.0,8.0,0.0,2.0,4.0,0.0,6.0,23.0,12,4,2,8,4,1,0,5.0,0.0,0,0.0,0.0,0.0,0.0,0,0,12.0,10.0,0.833333,136.0,9.0,9.0,2,1,0.907407,0.666667,1.000000,0.833333,1.000000,0.750000,1.000000,0.1,7,322.0
3,52669,RR__Boa Vista,Norte,RR,D-E,Casa,0,0,4,0,2,2,0,49.0,1,1,0,2,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,6,2,250.0,30000.0,35.0,0.0,5.0,0.0,5.0,4.0,0.0,0.0,22.0,12,0,0,1,0,0,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0,9.0,3.0,0.333333,136.0,12.0,9.0,0,1,0.685185,0.666667,0.666667,0.833333,0.600000,0.500000,1.000000,0.1,7,56.0
4,52687,RR__Boa Vista,Norte,RR,D-E,Casa,0,0,3,0,1,2,0,14.0,0,1,0,1,2,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,7,2,150.0,11000.0,45.0,0.0,4.0,0.0,3.0,3.0,0.0,0.0,16.0,11,0,0,4,3,0,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0,6.0,1.0,0.166667,310.0,17.0,13.0,0,1,0.407407,0.666667,1.000000,0.833333,0.133333,0.083333,0.777778,0.1,7,192.0


In [13]:

# Resumo final para inspeção rápida.
resumo = pd.DataFrame([
    {
        "periodo": periodo,
        "linhas": len(dataframe),
        "colunas": dataframe.shape[1],
        "nulos": int(dataframe.isna().sum().sum()),
        "consumo_medio_kwh_media": dataframe["consumo_medio_kwh"].mean(),
        "consumo_medio_kwh_mediana": dataframe["consumo_medio_kwh"].median(),
        "consumo_medio_kwh_min": dataframe["consumo_medio_kwh"].min(),
        "consumo_medio_kwh_max": dataframe["consumo_medio_kwh"].max(),
    }
    for periodo, dataframe in {"jan_jun": df_jan_jun, "jun_dez": df_jun_dez}.items()
])
resumo


,periodo,linhas,colunas,nulos,consumo_medio_kwh_media,consumo_medio_kwh_mediana,consumo_medio_kwh_min,consumo_medio_kwh_max
0,jan_jun,2402,87,0,159.955239,134.833333,6.000000,1022.8
1,jun_dez,2548,87,0,163.464816,135.714286,20.714286,1001.0



## Separação recomendada no próximo notebook

```python
colunas_controle = [
    "id_entrevista",
    "grupo_validacao_municipio",
    "controle_meses_consumo_validos",
]
alvo = "consumo_medio_kwh"

# Escolha df_jan_jun ou df_jun_dez.
dataframe = df_jan_jun
X = dataframe.drop(columns=colunas_controle + [alvo])
y = dataframe[alvo]
grupos = dataframe["grupo_validacao_municipio"]
```

As colunas categóricas podem ser tratadas posteriormente com `OneHotEncoder`, `OrdinalEncoder` ou por um modelo com suporte nativo a categorias.
